## Standardising Fields and Labels

Inconsistent data values are a silent killer of analysis. If the same concept is recorded in different ways, your `GROUP BY` will split it into separate categories, your `JOIN` will fail to match, and your counts will be wrong.

> **Note:** All data in this module is **entirely synthetic** and does not represent any real schools, pupils, or individuals.

We'll use `schools_autumn_2024` from `catalog_40_copper_analyst_training.bronze`, which contains inconsistencies in casing, whitespace, abbreviations, and null handling.

### Common problems

| Problem | Example | Impact |
| --- | --- | --- |
| **Case differences** | `'Academy'` vs `'academy'` vs `'Acad'` | Multiple groups instead of one |
| **Whitespace** | `' Ash Valley School '` vs `'Ash Valley School'` | Joins silently fail |
| **Abbreviations** | `'Acad'` vs `'Academy'`, `'Sec'` vs `'Secondary'` | Incomplete grouping |
| **Null vs empty string** | `NULL` vs `''` vs `'N/A'` | Inconsistent null handling |

### Standardisation techniques

* **`LOWER()` or `UPPER()`** — normalise case
* **`TRIM()`** — remove leading/trailing whitespace
* **`REPLACE()` / `REGEXP_REPLACE()`** — fix known abbreviations or encoding issues
* **`COALESCE()`** — unify nulls and empty strings
* **`CASE WHEN` / lookup tables** — map variant labels to a canonical form
* **`TO_DATE()` / `DATE_FORMAT()`** — parse and standardise dates

In [0]:
-- Preview the messy schools data — look for inconsistencies in school_type, city, phase
SELECT
  school_urn
  ,school_name
  ,city
  ,school_type
  ,phase
FROM catalog_40_copper_analyst_training.bronze.schools_autumn_2024
ORDER BY school_urn;

In [0]:
-- Basic standardisation: clean up the most common issues manually
SELECT
  school_urn

  -- Trim whitespace and apply title case
  ,INITCAP(TRIM(school_name)) as school_name_clean

  -- Standardise city: trim, lowercase then title case
  ,INITCAP(TRIM(LOWER(city))) as city_clean

  -- Map school_type variants to canonical labels
  ,CASE
    WHEN LOWER(TRIM(COALESCE(school_type, ''))) IN ('academy', 'acad')
      THEN 'Academy'
    WHEN LOWER(TRIM(COALESCE(school_type, ''))) = 'maintained'
      THEN 'Maintained'
    WHEN LOWER(TRIM(COALESCE(school_type, ''))) = 'free school'
      THEN 'Free School'
    ELSE 'Unknown'
  END as school_type_clean

  -- Standardise phase: handle abbreviations and nulls
  ,CASE
    WHEN LOWER(TRIM(COALESCE(phase, ''))) IN ('secondary', 'sec')
      THEN 'Secondary'
    WHEN LOWER(TRIM(COALESCE(phase, ''))) = 'primary'
      THEN 'Primary'
    WHEN LOWER(TRIM(COALESCE(phase, ''))) = 'all-through'
      THEN 'All-through'
    ELSE 'Unknown'
  END as phase_clean

FROM catalog_40_copper_analyst_training.bronze.schools_autumn_2024
ORDER BY school_urn;

In [0]:
-- Using a lookup table for more maintainable standardisation
-- This approach scales better when many variants exist and is easier to audit

CREATE OR REPLACE TEMP VIEW school_type_lookup AS
SELECT * FROM VALUES
  ('academy', 'Academy')
  ,('acad', 'Academy')
  ,('maintained', 'Maintained')
  ,('la maintained', 'Maintained')
  ,('free school', 'Free School')
  ,('n/a', 'Unknown')
AS t(raw_value, canonical_value);

-- Join to the lookup to standardise
SELECT
  s.school_urn
  ,INITCAP(TRIM(s.school_name)) as school_name_clean
  ,INITCAP(TRIM(LOWER(s.city))) as city_clean
  ,COALESCE(lu.canonical_value, 'Unknown') as school_type_clean
FROM catalog_40_copper_analyst_training.bronze.schools_autumn_2024 s
LEFT JOIN school_type_lookup lu
  ON LOWER(TRIM(COALESCE(s.school_type, ''))) = lu.raw_value
ORDER BY s.school_urn;

### Going further: discovering inconsistencies dynamically

The manual approach requires you to already know which fields have problems. For an unfamiliar dataset, you can use `INFORMATION_SCHEMA` to **find all string columns** and then query distinct values for each one, surfacing inconsistencies automatically.

This is particularly useful for data profiling — a first pass to understand the quality of a new dataset before deciding which fields need standardisation.

In [0]:
-- Step 1: Find all string columns in the table using INFORMATION_SCHEMA
SELECT
  column_name
  ,data_type
FROM catalog_40_copper_analyst_training.information_schema.columns
WHERE table_schema = 'bronze'
  AND table_name = 'schools_autumn_2024'
  AND data_type = 'STRING'
ORDER BY ordinal_position;

In [0]:
-- Step 2: Dynamically profile each string column
-- For each string column, show the distinct values and their counts
-- This instantly surfaces casing issues, whitespace, abbreviations, and nulls

DECLARE profile_sql STRING;

SET VAR profile_sql = (
  WITH str_cols AS (
    SELECT column_name
    FROM catalog_40_copper_analyst_training.information_schema.columns
    WHERE table_schema = 'bronze'
      AND table_name = 'schools_autumn_2024'
      AND data_type = 'STRING'
      AND column_name != 'metadata_json'
    ORDER BY ordinal_position
  )
  SELECT aggregate(
    collect_list(
      concat(
        'SELECT ''', column_name, ''' as column_name, '
        ,'CAST(`', column_name, '` AS STRING) as raw_value, '
        ,'LOWER(TRIM(COALESCE(CAST(`', column_name, '` AS STRING), ''[NULL]''))) as normalised_value, '
        ,'COUNT(*) as row_count '
        ,'FROM catalog_40_copper_analyst_training.bronze.schools_autumn_2024 '
        ,'GROUP BY 1, 2, 3'
      )
    )
    ,''
    ,(acc, x) -> CASE WHEN acc = '' THEN x ELSE concat(acc, ' UNION ALL ', x) END
  )
  FROM str_cols
);

EXECUTE IMMEDIATE concat(profile_sql, ' ORDER BY column_name, normalised_value, raw_value');

> **Reading the output:** Look for rows where the `column_name` and `normalised_value` are the same but `raw_value` differs — these are the inconsistencies that need standardising. For example, you might see `raw_value` entries of `'Academy'`, `'academy'`, and `'Acad'` all mapping to similar normalised values, confirming they need to be unified.